In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [ ]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

### Let's divide the training and test set into two partitions

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training (70%) and validation (30%)
data_train, data_val = train_test_split(data, test_size=0.3, random_state=42, stratify=data['label'])

print(f"Training set size: {len(data_train)}")
print(f"Validation set size: {len(data_val)}")
print(f"\nTraining set label distribution:\n{data_train['label'].value_counts()}")
print(f"\nValidation set label distribution:\n{data_val['label'].value_counts()}")

## Data Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [ ]:
import re

# Remove inline JavaScript and CSS
def remove_inline_code(text):
    # Remove <script>...</script> tags
    text = re.sub(r'<script[^>]*>.*?</script>', '', text, flags=re.IGNORECASE | re.DOTALL)
    # Remove <style>...</style> tags
    text = re.sub(r'<style[^>]*>.*?</style>', '', text, flags=re.IGNORECASE | re.DOTALL)
    # Remove event handlers (onclick, onload, etc.)
    text = re.sub(r'\s*on\w+\s*=\s*["\'][^"\']*["\']', '', text, flags=re.IGNORECASE)
    # Remove javascript: URLs
    text = re.sub(r'javascript:', '', text, flags=re.IGNORECASE)
    return text

data['preprocessed_text'] = data['text'].apply(remove_inline_code)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [ ]:
def clean_text(text):
    # Remove all special characters (keep only alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove all single characters
    text = re.sub(r'\b\w\b', '', text)
    
    # Remove single characters from the start
    text = re.sub(r'^\w\s', '', text)
    
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove prefixed 'b' (like b'...' from byte strings)
    text = re.sub(r'\bb\'', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    return text.strip()

data['preprocessed_text'] = data['preprocessed_text'].apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)

data['preprocessed_text'] = data['preprocessed_text'].apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)

data['preprocessed_text'] = data['preprocessed_text'].apply(lemmatize_text)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
from collections import Counter

# Separate ham and spam messages
ham_messages = data[data['label'] == 'ham']['preprocessed_text'].str.cat(sep=' ')
spam_messages = data[data['label'] == 'spam']['preprocessed_text'].str.cat(sep=' ')

# Count word frequencies
ham_words = Counter(ham_messages.split())
spam_words = Counter(spam_messages.split())

# Get top 10 words
top_10_ham = ham_words.most_common(10)
top_10_spam = spam_words.most_common(10)

print("Top 10 words in HAM messages:")
print(top_10_ham)
print("\nTop 10 words in SPAM messages:")
print(top_10_spam)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ham words plot
ham_words_list, ham_counts = zip(*top_10_ham)
axes[0].barh(ham_words_list, ham_counts, color='green', alpha=0.7)
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 10 Words in HAM Messages')
axes[0].invert_yaxis()

# Spam words plot
spam_words_list, spam_counts = zip(*top_10_spam)
axes[1].barh(spam_words_list, spam_counts, color='red', alpha=0.7)
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top 10 Words in SPAM Messages')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Extra features

In [ ]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

## How would work the Bag of Words with Count Vectorizer concept?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create CountVectorizer - converts text to word count matrix
count_vectorizer = CountVectorizer(max_features=100)  # Limit to top 100 words

# Fit and transform training data
X_train_bow = count_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bow = count_vectorizer.transform(data_val['preprocessed_text'])

print(f"Bag of Words Matrix Shape (train): {X_train_bow.shape}")
print(f"Bag of Words Matrix Shape (val): {X_val_bow.shape}")
print(f"\nNumber of features (unique words): {len(count_vectorizer.get_feature_names_out())}")
print(f"\nFirst 20 features (words):\n{count_vectorizer.get_feature_names_out()[:20]}")

# Show example: first document as dense array
print(f"\nExample - First document word counts:\n{X_train_bow[0].toarray()}")
print(f"\nNon-zero words in first document:")
feature_names = count_vectorizer.get_feature_names_out()
nonzero_idx = X_train_bow[0].nonzero()[1]
for idx in nonzero_idx:
    word = feature_names[idx]
    count = X_train_bow[0, idx]
    print(f"  '{word}': {count}")

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [ ]:
# Load the vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100)  # Limit to top 100 words

# Fit on training data and transform both sets
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])

# Print the shape of the vectorized dataset
print(f"TF-IDF Matrix Shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF Matrix Shape (val): {X_val_tfidf.shape}")
print(f"\nNumber of features (unique words): {len(tfidf_vectorizer.get_feature_names_out())}")

# Show example: TF-IDF scores for first document
print(f"\nExample - First document TF-IDF scores:")
feature_names = tfidf_vectorizer.get_feature_names_out()
nonzero_idx = X_train_tfidf[0].nonzero()[1]
tfidf_scores = list(zip(feature_names[nonzero_idx], X_train_tfidf[0].data))
# Sort by TF-IDF score descending
tfidf_scores.sort(key=lambda x: x[1], reverse=True)
for word, score in tfidf_scores[:10]:
    print(f"  '{word}': {score:.4f}")

## And the Train a Classifier?

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Create and train the classifier (using TF-IDF vectors)
classifier = MultinomialNB()
classifier.fit(X_train_tfidf, data_train['label'])

# Make predictions
y_train_pred = classifier.predict(X_train_tfidf)
y_val_pred = classifier.predict(X_val_tfidf)

# Evaluate on training set
train_accuracy = accuracy_score(data_train['label'], y_train_pred)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Training Precision: {precision_score(data_train['label'], y_train_pred, average='weighted'):.4f}")
print(f"Training Recall: {recall_score(data_train['label'], y_train_pred, average='weighted'):.4f}")
print(f"Training F1-Score: {f1_score(data_train['label'], y_train_pred, average='weighted'):.4f}")

# Evaluate on validation set
val_accuracy = accuracy_score(data_val['label'], y_val_pred)
print(f"\nValidation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {precision_score(data_val['label'], y_val_pred, average='weighted'):.4f}")
print(f"Validation Recall: {recall_score(data_val['label'], y_val_pred, average='weighted'):.4f}")
print(f"Validation F1-Score: {f1_score(data_val['label'], y_val_pred, average='weighted'):.4f}")

# Confusion matrix
print(f"\nConfusion Matrix (Validation):")
print(confusion_matrix(data_val['label'], y_val_pred))

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
import numpy as np

# Dictionary to store results
results = {}

# Test 1: Bag of Words only
print("=" * 60)
print("TEST 1: Bag of Words Only")
print("=" * 60)
clf1 = MultinomialNB()
clf1.fit(X_train_bow, data_train['label'])
y_pred1 = clf1.predict(X_val_bow)
acc1 = accuracy_score(data_val['label'], y_pred1)
f1_1 = f1_score(data_val['label'], y_pred1, average='weighted')
results['BoW Only'] = {'accuracy': acc1, 'f1': f1_1}
print(f"Validation Accuracy: {acc1:.4f}")
print(f"Validation F1-Score: {f1_1:.4f}\n")

# Test 2: TF-IDF only
print("=" * 60)
print("TEST 2: TF-IDF Only")
print("=" * 60)
clf2 = MultinomialNB()
clf2.fit(X_train_tfidf, data_train['label'])
y_pred2 = clf2.predict(X_val_tfidf)
acc2 = accuracy_score(data_val['label'], y_pred2)
f1_2 = f1_score(data_val['label'], y_pred2, average='weighted')
results['TF-IDF Only'] = {'accuracy': acc2, 'f1': f1_2}
print(f"Validation Accuracy: {acc2:.4f}")
print(f"Validation F1-Score: {f1_2:.4f}\n")

# Test 3: Bag of Words + Extra Flags
print("=" * 60)
print("TEST 3: Bag of Words + Extra Flags")
print("=" * 60)
X_train_bow_extra = np.hstack([X_train_bow.toarray(), data_train[['money_mark', 'suspicious_words', 'text_len']].values])
X_val_bow_extra = np.hstack([X_val_bow.toarray(), data_val[['money_mark', 'suspicious_words', 'text_len']].values])
clf3 = MultinomialNB()
clf3.fit(X_train_bow_extra, data_train['label'])
y_pred3 = clf3.predict(X_val_bow_extra)
acc3 = accuracy_score(data_val['label'], y_pred3)
f1_3 = f1_score(data_val['label'], y_pred3, average='weighted')
results['BoW + Flags'] = {'accuracy': acc3, 'f1': f1_3}
print(f"Validation Accuracy: {acc3:.4f}")
print(f"Validation F1-Score: {f1_3:.4f}\n")

# Test 4: TF-IDF + Extra Flags
print("=" * 60)
print("TEST 4: TF-IDF + Extra Flags")
print("=" * 60)
X_train_tfidf_extra = np.hstack([X_train_tfidf.toarray(), data_train[['money_mark', 'suspicious_words', 'text_len']].values])
X_val_tfidf_extra = np.hstack([X_val_tfidf.toarray(), data_val[['money_mark', 'suspicious_words', 'text_len']].values])
clf4 = MultinomialNB()
clf4.fit(X_train_tfidf_extra, data_train['label'])
y_pred4 = clf4.predict(X_val_tfidf_extra)
acc4 = accuracy_score(data_val['label'], y_pred4)
f1_4 = f1_score(data_val['label'], y_pred4, average='weighted')
results['TF-IDF + Flags'] = {'accuracy': acc4, 'f1': f1_4}
print(f"Validation Accuracy: {acc4:.4f}")
print(f"Validation F1-Score: {f1_4:.4f}\n")

# Summary and Recommendation
print("=" * 60)
print("SUMMARY - COMPARISON OF ALL MODELS")
print("=" * 60)
results_df = pd.DataFrame(results).T
print(results_df)
print()
best_model = results_df['accuracy'].idxmax()
best_accuracy = results_df['accuracy'].max()
print(f"🏆 BEST MODEL: {best_model}")
print(f"   Accuracy: {best_accuracy:.4f}")
print(f"   F1-Score: {results_df.loc[best_model, 'f1']:.4f}")